#训练FP32 CNN model

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

# ==========================================
# 1. 定义硬件友好的极简 CNN (全 FP32)
# ==========================================
class HardwareFriendlyCNN(nn.Module):
    def __init__(self):
        super(HardwareFriendlyCNN, self).__init__()
        # 只使用基础算子，方便后续转纯 C 语言和 NPU 硬件实现
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        # CIFAR-10 尺寸: 32x32 -> pool -> 16x16 -> pool -> 8x8
        self.fc = nn.Linear(32 * 8 * 8, 10) 
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x

def main():
    # 检测是否能用 GPU 加速训练
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    # ==========================================
    # 2. 准备 CIFAR-10 数据集 (自动下载)
    # ==========================================
    transform = transforms.Compose([
        transforms.ToTensor(), # 将图片转为 [0.0, 1.0] 的浮点数 Tensor
        # 这里为了简化硬件 C 代码的预处理，我们省略了 Normalize(均值方差归一化)
    ])

    print("Loading datasets...")
    trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
    trainloader = torch.utils.data.DataLoader(trainset, batch_size=64, shuffle=True)

    testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
    testloader = torch.utils.data.DataLoader(testset, batch_size=1000, shuffle=False)

    # ==========================================
    # 3. 训练模型 (这一步将生成属于你的开源权重)
    # ==========================================
    model = HardwareFriendlyCNN().to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=15, gamma=0.5)

    epochs = 50 # 跑 10 轮大概需要 2~3 分钟
    print(f"Starting training for {epochs} epochs...")
    
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for inputs, labels in trainloader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
        scheduler.step()
        print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss/len(trainloader):.4f}, LR: {scheduler.get_last_lr()[0]:.6f}")

    # 保存属于你的 FP32 黄金模型权重
    torch.save(model.state_dict(), 'my_hardware_cnn_fp32.pth')
    print("Model weights saved to 'my_hardware_cnn_fp32.pth'")

    # ==========================================
    # 4. 在测试集上进行全局准确率验证
    # ==========================================
    print("Testing FP32 model accuracy on 10000 test images...")
    model.eval() # 切换到推理模式
    correct = 0
    total = 0
    
    with torch.no_grad(): # 推理时不需要计算梯度
        for inputs, labels in testloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1) # 获取得分最大的类别
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f"FP32 Model Accuracy on the 10000 test images: {accuracy:.2f} %")

if __name__ == '__main__':
    main()

Using device: cuda
Loading datasets...
Files already downloaded and verified
Files already downloaded and verified
Starting training for 50 epochs...
Epoch 1/50, Loss: 1.5794, LR: 0.001000
Epoch 2/50, Loss: 1.2743, LR: 0.001000
Epoch 3/50, Loss: 1.1455, LR: 0.001000
Epoch 4/50, Loss: 1.0517, LR: 0.001000
Epoch 5/50, Loss: 0.9895, LR: 0.001000
Epoch 6/50, Loss: 0.9415, LR: 0.001000
Epoch 7/50, Loss: 0.9016, LR: 0.001000
Epoch 8/50, Loss: 0.8775, LR: 0.001000
Epoch 9/50, Loss: 0.8520, LR: 0.001000
Epoch 10/50, Loss: 0.8282, LR: 0.001000
Epoch 11/50, Loss: 0.8142, LR: 0.001000
Epoch 12/50, Loss: 0.7982, LR: 0.001000
Epoch 13/50, Loss: 0.7801, LR: 0.001000
Epoch 14/50, Loss: 0.7660, LR: 0.001000
Epoch 15/50, Loss: 0.7540, LR: 0.000500
Epoch 16/50, Loss: 0.7076, LR: 0.000500
Epoch 17/50, Loss: 0.6994, LR: 0.000500
Epoch 18/50, Loss: 0.6935, LR: 0.000500
Epoch 19/50, Loss: 0.6860, LR: 0.000500
Epoch 20/50, Loss: 0.6811, LR: 0.000500
Epoch 21/50, Loss: 0.6746, LR: 0.000500
Epoch 22/50, Loss: 